In [1]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split

## Process the dataset by keeping the positive impressions as the "items" we want to predict

In [2]:
# --- Load and preprocess ---
# Load behaviors.tsv
behaviors_df = pd.read_csv('MINDsmall_train/behaviors.tsv', sep='\t', header=None,
                           names=["impression_id", "user_id", "time", "history", "impressions"])

# Convert history to list
behaviors_df['history'] = behaviors_df['history'].fillna('').apply(lambda x: x.split())

# Convert impressions to list of (news_id, clicked)
def parse_impressions(imp):
    return [(x.split('-')[0], int(x.split('-')[1])) for x in imp.split()]

behaviors_df['impressions'] = behaviors_df['impressions'].apply(parse_impressions)

# --- Extract unique users and split them with ratio 8:1:1 ---
unique_users = behaviors_df['user_id'].unique()
print(f"Total unique users: {len(unique_users)}")
train_users, val_test_users = train_test_split(unique_users, test_size=0.2, random_state=42)
val_users, test_users = train_test_split(val_test_users, test_size=0.5, random_state=42)

# --- Set of train, val, test users ---
train_users_set = set(train_users)
val_users_set = set(val_users)
test_users_set = set(test_users)

# --- Create train, val, test dataframes ---
records_train = []
records_val = []
records_test = []

for _, row in behaviors_df.iterrows():
    uid = row['user_id']
    history = row['history']
    # Ensure history is not empty with eval 
    if not history:
        continue
    if uid in train_users_set:
        for news_id, clicked in row['impressions']:
            if clicked == 1:
                records_train.append({'user_id': uid, 'history': ' '.join(history), 'clicked_news': news_id})
    elif uid in val_users_set:
        for news_id, clicked in row['impressions']:
            if clicked == 1:
                records_val.append({'user_id': uid, 'history': ' '.join(history), 'clicked_news': news_id})
    elif uid in test_users_set:
        for news_id, clicked in row['impressions']:
            if clicked == 1:
                records_test.append({'user_id': uid, 'history': ' '.join(history), 'clicked_news': news_id})

# Convert lists to DataFrame
train_df = pd.DataFrame(records_train)
val_df = pd.DataFrame(records_val)
test_df = pd.DataFrame(records_test)
#print(f"Total positive clicks: {len(click_df)}")

# Group val_df by user_id and history, aggregating clicked_news into space-separated string
val_df_grouped = val_df.groupby(['user_id', 'history'])['clicked_news'] \
                       .apply(lambda x: ' '.join(x)) \
                       .reset_index()

# Same for test_df
test_df_grouped = test_df.groupby(['user_id', 'history'])['clicked_news'] \
                         .apply(lambda x: ' '.join(x)) \
                         .reset_index()

Total unique users: 50000


In [3]:
# # Group train_df by user_id and history, aggregating clicked_news into space-separated string
# train_df_grouped = train_df.groupby(['user_id', 'history'])['clicked_news'] \
#                            .apply(lambda x: ' '.join(x)) \
#                            .reset_index()

# Concatenate clicked_news into history
train_df_grouped=train_df.copy()
train_df_grouped['history'] = train_df['history'] + ' ' + train_df['clicked_news']
train_df_grouped.drop(columns='clicked_news', inplace=True)


In [4]:
# print sorted by user_id
train_df_grouped.sort_values(by='user_id', inplace=True)
train_df_grouped.head()

,user_id,history
54954,U1000,N29641 N1789 N41244 N7670
102574,U1000,N29641 N1789 N41244 N53875
87944,U1000,N29641 N1789 N41244 N58656
54953,U1000,N29641 N1789 N41244 N29739
184664,U10001,N47937 N51706 N56447 N61319 N27644 N18030 N272...


In [5]:
train_df.sort_values(by='user_id', inplace=True)
train_df.head(5)

,user_id,history,clicked_news
54954,U1000,N29641 N1789 N41244,N7670
102574,U1000,N29641 N1789 N41244,N53875
87944,U1000,N29641 N1789 N41244,N58656
54953,U1000,N29641 N1789 N41244,N29739
184664,U10001,N47937 N51706 N56447 N61319 N27644 N18030 N272...,N35937


In [6]:
print(train_df_grouped.head())
print(val_df_grouped.head())
print(test_df_grouped.head())

       user_id                                            history
54954    U1000                          N29641 N1789 N41244 N7670
102574   U1000                         N29641 N1789 N41244 N53875
87944    U1000                         N29641 N1789 N41244 N58656
54953    U1000                         N29641 N1789 N41244 N29739
184664  U10001  N47937 N51706 N56447 N61319 N27644 N18030 N272...
  user_id                                            history  \
0  U10014                         N24002 N25292 N1919 N19660   
1  U10025  N26642 N44559 N32483 N6233 N56446 N36526 N564 ...   
2  U10079  N26618 N24593 N19495 N11032 N14349 N54536 N110...   
3  U10101  N14629 N47558 N2203 N59173 N28082 N14102 N3669...   
4  U10111  N55189 N44559 N41944 N51706 N16887 N38246 N407...   

                                        clicked_news  
0                                      N63656 N35170  
1  N59981 N24473 N57426 N45016 N31947 N37079 N605...  
2                                      N23446 N50872  

In [7]:
# --- Step 1: Collect all unique user_ids ---
all_user_ids = pd.concat([train_df_grouped['user_id'], val_df_grouped['user_id'], test_df_grouped['user_id']]).unique()

In [8]:
print(all_user_ids[:10])  # Print first 10 unique user IDs
print(sorted(all_user_ids[:10]))  # Print first 10 unique user IDs
user2int = {uid: idx for idx, uid in enumerate(sorted(all_user_ids))}
print(user2int)  # Print the user to integer mapping

['U1000' 'U10001' 'U10003' 'U10010' 'U10011' 'U10012' 'U10016' 'U10017'
 'U1002' 'U10022']
['U1000', 'U10001', 'U10003', 'U10010', 'U10011', 'U10012', 'U10016', 'U10017', 'U1002', 'U10022']
{'U100': 0, 'U1000': 1, 'U10001': 2, 'U10003': 3, 'U10008': 4, 'U10010': 5, 'U10011': 6, 'U10012': 7, 'U10014': 8, 'U10015': 9, 'U10016': 10, 'U10017': 11, 'U1002': 12, 'U10022': 13, 'U10023': 14, 'U10024': 15, 'U10025': 16, 'U10026': 17, 'U10027': 18, 'U1003': 19, 'U10030': 20, 'U10032': 21, 'U10034': 22, 'U10038': 23, 'U10039': 24, 'U1004': 25, 'U10040': 26, 'U10042': 27, 'U10043': 28, 'U10044': 29, 'U10045': 30, 'U10046': 31, 'U10047': 32, 'U10048': 33, 'U10049': 34, 'U10050': 35, 'U10051': 36, 'U10052': 37, 'U10053': 38, 'U10055': 39, 'U10056': 40, 'U10058': 41, 'U10059': 42, 'U1006': 43, 'U10060': 44, 'U10061': 45, 'U10062': 46, 'U10063': 47, 'U10064': 48, 'U10065': 49, 'U10066': 50, 'U10070': 51, 'U10071': 52, 'U10072': 53, 'U10075': 54, 'U10076': 55, 'U10077': 56, 'U10078': 57, 'U10079': 58, 

In [9]:
# --- Step 1: Collect all unique user_ids ---
all_user_ids = pd.concat([train_df_grouped['user_id'], val_df_grouped['user_id'], test_df_grouped['user_id']]).unique()
user2int = {uid: idx for idx, uid in enumerate(sorted(all_user_ids))}

# --- Step 2: Collect all unique news_ids from history and clicked_news ---
def extract_news_ids(df, history_col='history', clicked_col='clicked_news', train=False):
    history_news = df[history_col].str.split().explode()
    if not train:
        clicked_news = df[clicked_col].str.split().explode()
        return pd.concat([history_news, clicked_news]).dropna().unique()
    else:
        return history_news.dropna().unique()

# Use np.concatenate instead of pd.concat since we are handling arrays
all_news_ids = np.concatenate([
    extract_news_ids(train_df_grouped, train=True),
    extract_news_ids(val_df_grouped),
    extract_news_ids(test_df_grouped)
])

all_news_ids = np.unique(all_news_ids)

news2int = {nid: idx for idx, nid in enumerate(sorted(all_news_ids))}

# --- Step 3: Replace user_ids and news_ids with integers ---
def map_df(df, user2int, news2int, train = False):
    df = df.copy()
    df['user_id'] = df['user_id'].map(user2int)
    if not train:
        df['history'] = df['history'].apply(lambda x: str([news2int[n] for n in x.split() if n in news2int]))
        df['clicked_news'] = df['clicked_news'].apply(lambda x: str([news2int[n] for n in x.split() if n in news2int]))
    else:
        df['history'] = df['history'].apply(lambda x: str([news2int[n] for n in x.split() if n in news2int]))

    return df

train_df = map_df(train_df_grouped, user2int, news2int, train=True)
val_df_grouped = map_df(val_df_grouped, user2int, news2int)
test_df_grouped = map_df(test_df_grouped, user2int, news2int)

# Rename the columns to match the expected format
train_df_grouped.rename(columns={'user_id': 'user_id', 'history': 'user_history'}, inplace=True)
val_df_grouped.rename(columns={'user_id': 'user_id', 'history': 'given_user_history', 'clicked_news': 'predicting_items'}, inplace=True)
test_df_grouped.rename(columns={'user_id': 'user_id', 'history': 'given_user_history', 'clicked_news': 'predicting_items'}, inplace=True)

# --- (Optional) Save mappings for reference ---
pd.Series(user2int).to_csv("dataset/user2idx.tsv", sep="\t", header=False)
pd.Series(news2int).to_csv("dataset/news2idx.tsv", sep="\t", header=False)

# --- Save final datasets ---
train_df_grouped.to_csv('dataset/training.tsv', sep='\t', index=False)
val_df_grouped.to_csv('dataset/validation.tsv', sep='\t', index=False)
test_df_grouped.to_csv('dataset/test.tsv', sep='\t', index=False)

print("Saved: training.tsv, validation.tsv, test.tsv, and mappings")

Saved: training.tsv, validation.tsv, test.tsv, and mappings


In [10]:
# Confirmation prints
print(f"Saved {len(train_df)} training samples")
print(f"Saved {len(val_df_grouped)} validation samples")
print(f"Saved {len(test_df_grouped)} test samples")

Saved 185909 training samples
Saved 4905 validation samples
Saved 4900 test samples


In [11]:
train_df_grouped.head()


,user_id,user_history
54954,U1000,N29641 N1789 N41244 N7670
102574,U1000,N29641 N1789 N41244 N53875
87944,U1000,N29641 N1789 N41244 N58656
54953,U1000,N29641 N1789 N41244 N29739
184664,U10001,N47937 N51706 N56447 N61319 N27644 N18030 N272...


In [12]:
val_df_grouped.head()

,user_id,given_user_history,predicting_items
0,8,"[9532, 10419, 6189, 6518]","[36408, 17073]"
1,16,"[11318, 23474, 15233, 35489, 31488, 18001, 314...","[33860, 9858, 32141, 23775, 14868, 18393, 3426..."
2,58,"[11297, 9942, 6398, 677, 2884, 30201, 677, 3805]","[9161, 27746]"
3,70,"[3075, 25505, 8188, 33314, 12272, 2712, 18126,...","[17143, 28213, 4924, 7973]"
4,80,"[30649, 23474, 21689, 28303, 4625, 19197, 2087...",[1079]


In [13]:
test_df_grouped.head()

,user_id,given_user_history,predicting_items
0,0,"[6852, 16253, 24414, 31021, 27224, 5974, 29491...",[38318]
1,4,"[9470, 13690, 24931, 15656, 39549, 18001, 2343...",[3620]
2,9,"[18214, 4434, 34664, 27003, 24300, 21538, 19720]",[37886]
3,35,"[29268, 23470, 1098, 23470, 24225, 4831, 33684...","[26651, 18817, 11946, 29421, 33632]"
4,56,"[38, 31540, 33691, 26059, 9729, 10995, 15642, ...","[1317, 31336]"


In [14]:
# load the tsv files
train_df = pd.read_csv('dataset_impressions_not_8_2/training.tsv', sep='\t')
val_df = pd.read_csv('dataset_impressions_not_8_2/validation.tsv', sep='\t')
test_df = pd.read_csv('dataset_impressions_not_8_2/test.tsv', sep='\t')
# Display the first few rows of each dataframe
print("Training Data:")
print(train_df.head())
print("\nValidation Data:")
print(val_df.head())    
print("\nTest Data:")
print(test_df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_impressions_not_8_2/training.tsv'

In [ ]:
# This code assumes train_df, val_df, and test_df are loaded as shown in the image.

# 1. Initialize a set to store all unique item IDs.
# A set is used because it automatically handles duplicates.
all_unique_items = set()

# 2. Add unique items from the 'next_item' column in train_df.
# This column contains single integer item IDs.
all_unique_items.update(train_df['next_item'].dropna().unique())

# 3. Create a helper function to parse columns containing string-formatted lists.
# e.g., a string like '[101, 202, 303]'
def parse_and_add_items(series, item_set):
    """
    Parses a pandas Series of string-formatted lists, extracts the integer items,
    and adds them to the provided set.
    """
    # Iterate over each string-list in the series, ignoring any missing values (NaNs).
    for str_list in series.dropna():
        # Remove the square brackets from the string.
        items_str = str_list.strip('[]')
        
        # If the resulting string is not empty (i.e., it wasn't just '[]'), process it.
        if items_str:
            # Split the string by ', ' to get individual item IDs as strings.
            # Convert each item ID to an integer and add them all to our master set.
            items = [int(i) for i in items_str.split(', ')]
            item_set.update(items)

# 4. Apply the helper function to all relevant columns across the dataframes.
parse_and_add_items(train_df['user_history'], all_unique_items)
parse_and_add_items(val_df['given_user_history'], all_unique_items)
parse_and_add_items(val_df['predicting_items'], all_unique_items)
parse_and_add_items(test_df['given_user_history'], all_unique_items)
parse_and_add_items(test_df['predicting_items'], all_unique_items)

# 5. Calculate the final count by getting the size of the set.
total_unique_item_count = len(all_unique_items)

print(f"The total number of unique items is: {total_unique_item_count}")

The total number of unique items is: 39815


## Process the dataset by not using the impressions at all and splitting the history of its user using the 80:20 ratio

In [47]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split

# --- Load and preprocess full dataset ---
behaviors_df = pd.read_csv(
    'MINDsmall_train/behaviors.tsv',
    sep='\t',
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)

# keep only user_id and history columns
behaviors_df = behaviors_df[['user_id', 'history']]
behaviors_df.drop_duplicates(inplace=True)

# Parse history: string -> list
behaviors_df['history'] = behaviors_df['history'].fillna('').apply(str.split)

# --- Map user_id and news_id to integers ---
all_user_ids = sorted(behaviors_df['user_id'].unique())
user2int = {uid: idx for idx, uid in enumerate(all_user_ids)}
behaviors_df['user_id'] = behaviors_df['user_id'].map(user2int)

all_news_ids = sorted(set(nid for sublist in behaviors_df['history'] for nid in sublist))
news2int = {nid: idx+1 for idx, nid in enumerate(all_news_ids)}

print(f"Total unique users: {len(user2int)}")
print(f"Total unique news items: {len(news2int)}")

# Convert history to list of int IDs
behaviors_df['history'] = behaviors_df['history'].apply(
    lambda x: [news2int[n] for n in x if n in news2int]
)

# --- Train/Val/Test user split ---
unique_users = np.array(behaviors_df['user_id'].unique())
train_users, val_test_users = train_test_split(unique_users, test_size=0.2, random_state=42)
val_users, test_users = train_test_split(val_test_users, test_size=0.5, random_state=42)

train_users_set = set(train_users)
val_users_set = set(val_users)
test_users_set = set(test_users)

print(f"Train users: {len(train_users_set)}")
print(f"Validation users: {len(val_users_set)}")
print(f"Test users: {len(test_users_set)}")

# --- Create all records (full set, with correct format) ---
train_records, val_records, test_records = [], [], []

append_train = train_records.append
append_val = val_records.append
append_test = test_records.append

for row in behaviors_df.itertuples(index=False):
    uid, history = row.user_id, row.history
    hlen = len(history)

    if hlen < 2:
        continue

    if uid in train_users_set:
        append_train(
            {
                'user_id': uid,
                'user_history': str(history),
            }
           
        )

    elif uid in val_users_set:
        split_idx = math.floor(hlen * 0.8)
        if 0 < split_idx < hlen:
            append_val({
                'user_id': uid,
                'given_user_history': str(history[:split_idx]),
                'predicting_items': str(history[split_idx:])
            })

    elif uid in test_users_set:
        split_idx = math.floor(hlen * 0.8)
        if 0 < split_idx < hlen:
            append_test({
                'user_id': uid,
                'given_user_history': str(history[:split_idx]),
                'predicting_items': str(history[split_idx:])
            })

# --- Convert to DataFrames ---
train_df = pd.DataFrame(train_records)
val_df = pd.DataFrame(val_records)
test_df = pd.DataFrame(test_records)


# --- Show Summary ---
print("\n--- Training Data Sample (Lower Triangular Logic) ---")
print(f"Total training examples created: {len(train_df)}")
print(train_df.head())

print("\n--- Validation Data Sample (80/20 Split) ---")
print(f"Total validation examples created: {len(val_df)}")
print(val_df.head())

print("\n--- Test Data Sample (80/20 Split) ---")
print(f"Total test examples created: {len(test_df)}")
print(test_df.head())


Total unique users: 50000
Total unique news items: 33195
Train users: 40000
Validation users: 5000
Test users: 5000

--- Training Data Sample (Lower Triangular Logic) ---
Total training examples created: 38662
   user_id                                       user_history
0     2246  [25616, 18584, 14003, 20322, 4776, 30188, 230,...
1    48340  [12338, 28727, 30036, 7994, 14559, 18915, 3214...
2    37557  [411, 9024, 31844, 6344, 17642, 25180, 28506, ...
3    14625  [20287, 6896, 32484, 24853, 17801, 18799, 1305...
4     5795  [16514, 2429, 12824, 12669, 6880, 19420, 5118,...

--- Validation Data Sample (80/20 Split) ---
Total validation examples created: 4819
   user_id                                 given_user_history   
0    42080                                  [47, 26360, 2775]  \
1    43475  [32314, 3263, 2408, 27667, 4908, 8081, 6956, 5...   
2    17582  [11086, 508, 11009, 31456, 6832, 27845, 8175, ...   
3    25408  [25616, 3512, 23637, 2848, 489, 28284, 22745, ...   
4     3

In [48]:
import ast

def print_dataset_stats(name, df, history_col, predict_col=None):
    # Parse lists from string-formatted columns
    all_items = []
    for row in df.itertuples(index=False):
        history = ast.literal_eval(getattr(row, history_col))
        all_items.extend(history)
        if predict_col:
            predicted = ast.literal_eval(getattr(row, predict_col))
            all_items.extend(predicted)

    print(f"\n--- {name.upper()} SET STATS ---")
    print(f"Rows: {len(df)}")
    print(f"Unique users: {df['user_id'].nunique()}")
    print(f"Unique news items: {len(set(all_items))}")

# 📊 Print stats for all datasets
#print_dataset_stats("train", train_df, history_col='user_history')
print_dataset_stats("validation", val_df, history_col='given_user_history', predict_col='predicting_items')
print_dataset_stats("test", test_df, history_col='given_user_history', predict_col='predicting_items')



--- VALIDATION SET STATS ---
Rows: 4819
Unique users: 4819
Unique news items: 13385

--- TEST SET STATS ---
Rows: 4823
Unique users: 4823
Unique news items: 13256


In [49]:
# --- (Optional) Save mappings for reference ---
pd.Series(user2int).to_csv("dataset/user2idx.tsv", sep="\t", header=False)
pd.Series(news2int).to_csv("dataset/news2idx.tsv", sep="\t", header=False)

# --- Save final datasets ---
train_df.to_csv('dataset/training.tsv', sep='\t', index=False)
val_df.to_csv('dataset/validation.tsv', sep='\t', index=False)
test_df.to_csv('dataset/test.tsv', sep='\t', index=False)

print("Saved: training.tsv, validation.tsv, test.tsv, and mappings")

Saved: training.tsv, validation.tsv, test.tsv, and mappings


In [50]:
# Load the tsv files
train_df = pd.read_csv('dataset/training.tsv', sep='\t')
val_df = pd.read_csv('dataset/validation.tsv', sep='\t')
test_df = pd.read_csv('dataset/test.tsv', sep='\t')
# Display the first few rows of each dataframe
print("Training Data:")
print(train_df.head())
print("\nValidation Data:")
print(val_df.head())    
print("\nTest Data:")
print(test_df.head())
# Display the last few rows of each dataframe
print("\nLast few rows of Training Data:")
print(train_df.tail())
print("\nLast few rows of Validation Data:")
print(val_df.tail())
print("\nLast few rows of Test Data:")
print(test_df.tail())

Training Data:
   user_id                                       user_history
0     2246  [25616, 18584, 14003, 20322, 4776, 30188, 230,...
1    48340  [12338, 28727, 30036, 7994, 14559, 18915, 3214...
2    37557  [411, 9024, 31844, 6344, 17642, 25180, 28506, ...
3    14625  [20287, 6896, 32484, 24853, 17801, 18799, 1305...
4     5795  [16514, 2429, 12824, 12669, 6880, 19420, 5118,...

Validation Data:
   user_id                                 given_user_history   
0    42080                                  [47, 26360, 2775]  \
1    43475  [32314, 3263, 2408, 27667, 4908, 8081, 6956, 5...   
2    17582  [11086, 508, 11009, 31456, 6832, 27845, 8175, ...   
3    25408  [25616, 3512, 23637, 2848, 489, 28284, 22745, ...   
4     3188  [12376, 28815, 27275, 27581, 565, 1230, 11472,...   

                                  predicting_items  
0                                          [13469]  
1  [21270, 9172, 6221, 21024, 30939, 16655, 17327]  
2                              [4869, 6865, 1

In [51]:
uid = np.array([42080, 43475, 17582])  # Example user IDs to filter
step_next_items = val_df.loc[val_df['user_id'].isin(uid), 'predicting_items'].values
print(step_next_items)

['[13469]' '[21270, 9172, 6221, 21024, 30939, 16655, 17327]'
 '[4869, 6865, 18508]']


In [52]:
import ast

def validate_val_df(val_df):
    print("\n[Validation Precheck]")

    invalid_rows = 0
    total_rows = len(val_df)

    for i, row in val_df.iterrows():
        try:
            # Parse the list string into actual list (if not already a list)
            predicting_items = row['predicting_items']
            if isinstance(predicting_items, str):
                predicting_items = ast.literal_eval(predicting_items)

            if not isinstance(predicting_items, list):
                raise ValueError(f"Invalid type at row {i}")

            if len(predicting_items) == 0:
                print(f"[Warning] Empty ground truth at row {i} (user_id={row['user_id']})")
                invalid_rows += 1

        except Exception as e:
            print(f"[Error] Failed to parse row {i}: {e}")
            invalid_rows += 1

    print(f"✅ Checked {total_rows} validation rows.")
    print(f"❌ Found {invalid_rows} rows with invalid or empty ground truths.\n")

    assert (total_rows - invalid_rows) == len(val_df), \
        "Mismatch: Some validation rows are malformed. Please fix before training."

validate_val_df(val_df)



[Validation Precheck]
✅ Checked 4819 validation rows.
❌ Found 0 rows with invalid or empty ground truths.



### Old code from here on, do not use

In [ ]:
# --- Step 1: Collect all unique user_ids ---
all_user_ids = pd.concat([train_df['user_id'], val_df['user_id'], test_df['user_id']]).unique()

In [ ]:
print(all_user_ids[:10])  # Print first 10 unique user IDs
print(sorted(all_user_ids[:10]))  # Print first 10 unique user IDs
user2int = {uid: idx for idx, uid in enumerate(sorted(all_user_ids))}
print(user2int)  # Print the user to integer mapping

['U13740' 'U91836' 'U73700' 'U34670' 'U19739' 'U79199' 'U53231' 'U89744'
 'U10045' 'U92486']
['U10045', 'U13740', 'U19739', 'U34670', 'U53231', 'U73700', 'U79199', 'U89744', 'U91836', 'U92486']
{'U100': 0, 'U1000': 1, 'U10001': 2, 'U10003': 3, 'U10008': 4, 'U10010': 5, 'U10011': 6, 'U10012': 7, 'U10014': 8, 'U10015': 9, 'U10016': 10, 'U10017': 11, 'U1002': 12, 'U10022': 13, 'U10023': 14, 'U10024': 15, 'U10025': 16, 'U10026': 17, 'U10027': 18, 'U1003': 19, 'U10030': 20, 'U10032': 21, 'U10034': 22, 'U10038': 23, 'U10039': 24, 'U1004': 25, 'U10040': 26, 'U10042': 27, 'U10043': 28, 'U10044': 29, 'U10045': 30, 'U10046': 31, 'U10048': 32, 'U10049': 33, 'U10050': 34, 'U10051': 35, 'U10052': 36, 'U10053': 37, 'U10055': 38, 'U10056': 39, 'U10058': 40, 'U10059': 41, 'U1006': 42, 'U10060': 43, 'U10061': 44, 'U10062': 45, 'U10063': 46, 'U10064': 47, 'U10065': 48, 'U10066': 49, 'U10070': 50, 'U10071': 51, 'U10072': 52, 'U10075': 53, 'U10076': 54, 'U10077': 55, 'U10078': 56, 'U10079': 57, 'U10082': 

In [ ]:
# --- Step 1: Collect all unique user_ids ---
all_user_ids = pd.concat([train_df['user_id'], val_df['user_id'], test_df['user_id']]).unique()
user2int = {uid: idx for idx, uid in enumerate(sorted(all_user_ids))}

# --- Step 2: Collect all unique news_ids from history and clicked_news ---
def extract_news_ids(df, history_col='history', clicked_col='clicked_news'):
    history_news = df[history_col].str.split().explode()
    clicked_news = df[clicked_col].str.split().explode()
    return pd.concat([history_news, clicked_news]).dropna().unique()

# Use np.concatenate instead of pd.concat since we are handling arrays
all_news_ids = np.concatenate([
    extract_news_ids(train_df),
    extract_news_ids(val_df),
    extract_news_ids(test_df)
])

all_news_ids = np.unique(all_news_ids)

news2int = {nid: idx for idx, nid in enumerate(sorted(all_news_ids))}

# --- Step 3: Replace user_ids and news_ids with integers ---
def map_df(df, user2int, news2int, train = False):
    df = df.copy()
    df['user_id'] = df['user_id'].map(user2int)
    if not train:
        df['history'] = df['history'].apply(lambda x: str([news2int[n] for n in x.split() if n in news2int]))
        df['clicked_news'] = df['clicked_news'].apply(lambda x: str([news2int[n] for n in x.split() if n in news2int]))
    else:
        df['history'] = df['history'].apply(lambda x: str([news2int[n] for n in x.split() if n in news2int]))
        df['clicked_news'] = df['clicked_news'].apply(lambda x: ' '.join([str(news2int[n]) for n in x.split() if n in news2int]))

    return df

train_df = map_df(train_df, user2int, news2int, train=True)
val_df = map_df(val_df, user2int, news2int)
test_df= map_df(test_df, user2int, news2int)

# Rename the columns to match the expected format
train_df.rename(columns={'user_id': 'user_id', 'history': 'user_history', 'clicked_news': 'next_item'}, inplace=True)
val_df.rename(columns={'user_id': 'user_id', 'history': 'given_user_history', 'clicked_news': 'predicting_items'}, inplace=True)
test_df.rename(columns={'user_id': 'user_id', 'history': 'given_user_history', 'clicked_news': 'predicting_items'}, inplace=True)

# --- (Optional) Save mappings for reference ---
pd.Series(user2int).to_csv("dataset/user2idx.tsv", sep="\t", header=False)
pd.Series(news2int).to_csv("dataset/news2idx.tsv", sep="\t", header=False)

# --- Save final datasets ---
train_df.to_csv('dataset/training.tsv', sep='\t', index=False)
val_df.to_csv('dataset/validation.tsv', sep='\t', index=False)
test_df.to_csv('dataset/test.tsv', sep='\t', index=False)

print("Saved: training.tsv, validation.tsv, test.tsv, and mappings")

In [ ]:
# Confirmation prints
print(f"Saved {len(train_df)} training samples")
print(f"Saved {len(val_df)} validation samples")
print(f"Saved {len(test_df)} test samples")

Saved 185909 training samples
Saved 4905 validation samples
Saved 4900 test samples


In [ ]:
train_df.head()


,user_id,user_history,next_item
0,2201,"[30649, 22252, 16738, 24308, 5668, 36166, 274,...",30987
1,47474,"[14736, 34392, 35981, 9512, 17401, 22647, 3854...",4741
2,36880,"[477, 10742, 38176, 7538, 21123, 30140, 34137,...",9404
3,14380,"[24268, 8188, 38949, 29756, 21313, 22509, 1558...",26936
4,5680,"[19770, 2881, 15318, 15127, 8170, 23238, 6063,...",7561


In [ ]:
val_df.head()

,user_id,given_user_history,predicting_items
0,8,"[9532, 10419, 6189, 6518]","[36408, 17073]"
1,16,"[11318, 23474, 15233, 35489, 31488, 18001, 314...","[33860, 9858, 32141, 23775, 14868, 18393, 3426..."
2,58,"[11297, 9942, 6398, 677, 2884, 30201, 677, 3805]","[9161, 27746]"
3,70,"[3075, 25505, 8188, 33314, 12272, 2712, 18126,...","[17143, 28213, 4924, 7973]"
4,80,"[30649, 23474, 21689, 28303, 4625, 19197, 2087...",[1079]


In [ ]:
test_df.head()

,user_id,given_user_history,predicting_items
0,0,"[6852, 16253, 24414, 31021, 27224, 5974, 29491...",[38318]
1,4,"[9470, 13690, 24931, 15656, 39549, 18001, 2343...",[3620]
2,9,"[18214, 4434, 34664, 27003, 24300, 21538, 19720]",[37886]
3,35,"[29268, 23470, 1098, 23470, 24225, 4831, 33684...","[26651, 18817, 11946, 29421, 33632]"
4,56,"[38, 31540, 33691, 26059, 9729, 10995, 15642, ...","[1317, 31336]"


In [ ]:
# load the tsv files
train_df = pd.read_csv('dataset/training.tsv', sep='\t')
val_df = pd.read_csv('dataset/validation.tsv', sep='\t')
test_df = pd.read_csv('dataset/test.tsv', sep='\t')
# Display the first few rows of each dataframe
print("Training Data:")
print(train_df.head())
print("\nValidation Data:")
print(val_df.head())    
print("\nTest Data:")
print(test_df.head())

Training Data:
   user_id                                       user_history  next_item
0     2201  [30649, 22252, 16738, 24308, 5668, 36166, 274,...      30987
1    47474  [14736, 34392, 35981, 9512, 17401, 22647, 3854...       4741
2    36880  [477, 10742, 38176, 7538, 21123, 30140, 34137,...       9404
3    14380  [24268, 8188, 38949, 29756, 21313, 22509, 1558...      26936
4     5680  [19770, 2881, 15318, 15127, 8170, 23238, 6063,...       7561

Validation Data:
   user_id                                 given_user_history   
0        8                          [9532, 10419, 6189, 6518]  \
1       16  [11318, 23474, 15233, 35489, 31488, 18001, 314...   
2       58   [11297, 9942, 6398, 677, 2884, 30201, 677, 3805]   
3       70  [3075, 25505, 8188, 33314, 12272, 2712, 18126,...   
4       80  [30649, 23474, 21689, 28303, 4625, 19197, 2087...   

                                    predicting_items  
0                                     [36408, 17073]  
1  [33860, 9858, 32141, 2377

In [ ]:
# This code assumes train_df, val_df, and test_df are loaded as shown in the image.

# 1. Initialize a set to store all unique item IDs.
# A set is used because it automatically handles duplicates.
all_unique_items = set()

# 2. Add unique items from the 'next_item' column in train_df.
# This column contains single integer item IDs.
all_unique_items.update(train_df['next_item'].dropna().unique())

# 3. Create a helper function to parse columns containing string-formatted lists.
# e.g., a string like '[101, 202, 303]'
def parse_and_add_items(series, item_set):
    """
    Parses a pandas Series of string-formatted lists, extracts the integer items,
    and adds them to the provided set.
    """
    # Iterate over each string-list in the series, ignoring any missing values (NaNs).
    for str_list in series.dropna():
        # Remove the square brackets from the string.
        items_str = str_list.strip('[]')
        
        # If the resulting string is not empty (i.e., it wasn't just '[]'), process it.
        if items_str:
            # Split the string by ', ' to get individual item IDs as strings.
            # Convert each item ID to an integer and add them all to our master set.
            items = [int(i) for i in items_str.split(', ')]
            item_set.update(items)

# 4. Apply the helper function to all relevant columns across the dataframes.
parse_and_add_items(train_df['user_history'], all_unique_items)
parse_and_add_items(val_df['given_user_history'], all_unique_items)
parse_and_add_items(val_df['predicting_items'], all_unique_items)
parse_and_add_items(test_df['given_user_history'], all_unique_items)
parse_and_add_items(test_df['predicting_items'], all_unique_items)

# 5. Calculate the final count by getting the size of the set.
total_unique_item_count = len(all_unique_items)

print(f"The total number of unique items is: {total_unique_item_count}")

The total number of unique items is: 39815


In [ ]:
import pandas as pd
import ast

# Replace with your actual file name
val_df = pd.read_csv("dataset/validation.tsv", sep='\t')

# Convert stringified lists to actual Python lists
val_df['given_user_history'] = val_df['given_user_history'].apply(ast.literal_eval)
val_df['predicting_items'] = val_df['predicting_items'].apply(ast.literal_eval)

# Extract for checking
debug_pred = val_df['given_user_history'].tolist()
debug_truth = val_df['predicting_items'].tolist()


In [ ]:
import numpy as np

def check_prediction_alignment(pre, ground_truth):
    print(f"\n🔍 Checking prediction-target alignment...")
    print(f"  → Number of predictions: {len(pre)}")
    print(f"  → Number of ground-truth entries: {len(ground_truth)}")
    
    assert len(pre) == len(ground_truth), \
        f"❌ Length mismatch: {len(pre)} predictions vs {len(ground_truth)} ground truth"

    bad_indices = []
    for i in range(len(pre)):
        pred = pre[i]
        truth = ground_truth[i]

        if pred is None or len(pred) == 0:
            print(f"⚠️ Empty prediction at index {i}")
            bad_indices.append(i)
            continue

        if not isinstance(pred, (list, np.ndarray)):
            print(f"❌ Invalid type in pre[{i}]: {type(pred)}")
            bad_indices.append(i)
            continue

        if not isinstance(truth, (list, np.ndarray)):
            print(f"❌ Invalid type in ground_truth[{i}]: {type(truth)}")
            bad_indices.append(i)

    if bad_indices:
        print(f"\n🚨 Found {len(bad_indices)} problematic entries. Example:")
        for idx in bad_indices[:3]:
            print(f"  pre[{idx}] = {pre[idx]}")
            print(f"  ground_truth[{idx}] = {ground_truth[idx]}")
    else:
        print("✅ All entries look aligned and well-formed.")
check_prediction_alignment(debug_pred, debug_truth)


🔍 Checking prediction-target alignment...
  → Number of predictions: 15084
  → Number of ground-truth entries: 15084
✅ All entries look aligned and well-formed.


## USE ONLY HISTORY TRAIN+TEST CONCATENATED

## Process the dataset by using the positive impressions and concatenating them with the history, and splitting the history of each user using the 80:20 ratio

In [63]:
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split

In [64]:
def combine_history_and_impressions(row):
    """
    Parses positive impressions and concatenates them with existing history.
    """
    # Handle cases where history is missing (NaN)
    history = row['history'] if pd.notna(row['history']) else ''

    # Split the impressions string and find clicked articles (ending in '-1')
    # Then, strip the '-1' to get the pure article ID
    positive_impressions = [
        imp.split('-')[0] for imp in row['impressions'].split(' ') if imp.endswith('-1')
    ]

    # If there are no positive impressions, just return the original history
    if not positive_impressions:
        return history

    # Join the found positive impression IDs into a space-separated string
    positive_impressions_str = ' '.join(positive_impressions)
    
    # Combine history and the new positive impressions
    # This logic avoids adding an unnecessary space if the history is empty
    if history:
        return history + ' ' + positive_impressions_str
    else:
        return positive_impressions_str
    

# Isolate the positive clicks from each impression log
def get_positive_clicks(impressions_str):
    """Extracts only the IDs of clicked articles (ending in '-1')"""
    if pd.isna(impressions_str):
        return ''
    
    positive_clicks = [
        imp.split('-')[0] for imp in impressions_str.split(' ') if imp.endswith('-1')
    ]
    return ' '.join(positive_clicks)

# Group by user and their initial history, then aggregate the positive clicks
def aggregate_clicks(series):
    """Joins a series of strings, filtering out any empty ones."""
    return ' '.join(item for item in series if item)


# Combine the original history with the aggregated positive clicks
def create_full_history(row):
    """Concatenates the original history with all aggregated clicks for that group."""
    history = row['history']
    new_clicks = row['positive_clicks']
    
    if history and new_clicks:
        return history + ' ' + new_clicks
    # If original history is empty, just return the new clicks.
    # If new clicks are empty, just return the history.
    else:
        return history + new_clicks


In [65]:
# --- Load and preprocess full dataset ---
behaviors_df = pd.read_csv(
    'MINDsmall_train/behaviors.tsv',
    sep='\t',
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)

behaviors_df = behaviors_df[['user_id', 'history', 'impressions']]

# Handle cases where history might be missing (NaN)
behaviors_df['history'].fillna('', inplace=True)


behaviors_df['positive_clicks'] = behaviors_df['impressions'].apply(get_positive_clicks)

# The core of the solution: group by user/history and aggregate the new clicks.
# The .apply() will join the 'positive_clicks' strings from all rows in the group.
aggregated_df = behaviors_df.groupby(['user_id', 'history'])['positive_clicks'].apply(aggregate_clicks).reset_index()

aggregated_df['history'] = aggregated_df.apply(create_full_history, axis=1)

# 4. Final Cleanup: Select and rename columns to match the desired output
final_df = aggregated_df[['user_id', 'history']]

behaviors_df = final_df.copy()


# Parse history: string -> list
behaviors_df['history'] = behaviors_df['history'].fillna('').apply(str.split)

# --- Map user_id and news_id to integers ---
all_user_ids = sorted(behaviors_df['user_id'].unique())
user2int = {uid: idx for idx, uid in enumerate(all_user_ids)}
behaviors_df['user_id'] = behaviors_df['user_id'].map(user2int)

all_news_ids = sorted(set(nid for sublist in behaviors_df['history'] for nid in sublist))
news2int = {nid: idx+1 for idx, nid in enumerate(all_news_ids)}

print(f"Total unique users: {len(user2int)}")
print(f"Total unique news items: {len(news2int)}")

# Convert history to list of int IDs
behaviors_df['history'] = behaviors_df['history'].apply(
    lambda x: [news2int[n] for n in x if n in news2int]
)

# --- Train/Val/Test user split ---
unique_users = np.array(behaviors_df['user_id'].unique())
train_users, val_test_users = train_test_split(unique_users, test_size=0.2, random_state=42)
val_users, test_users = train_test_split(val_test_users, test_size=0.5, random_state=42)

train_users_set = set(train_users)
val_users_set = set(val_users)
test_users_set = set(test_users)

print(f"Train users: {len(train_users_set)}")
print(f"Validation users: {len(val_users_set)}")
print(f"Test users: {len(test_users_set)}")

# --- Create all records (full set, with correct format) ---
train_records, val_records, test_records = [], [], []

append_train = train_records.append
append_val = val_records.append
append_test = test_records.append

for row in behaviors_df.itertuples(index=False):
    uid, history = row.user_id, row.history
    hlen = len(history)

    if hlen < 2:
        continue

    if uid in train_users_set:
        append_train(
            {
                'user_id': uid,
                'user_history': str(history),
            }
        )

    elif uid in val_users_set:
        split_idx = math.floor(hlen * 0.8)
        if 0 < split_idx < hlen:
            append_val({
                'user_id': uid,
                'given_user_history': str(history[:split_idx]),
                'predicting_items': str(history[split_idx:])
            })

    elif uid in test_users_set:
        split_idx = math.floor(hlen * 0.8)
        if 0 < split_idx < hlen:
            append_test({
                'user_id': uid,
                'given_user_history': str(history[:split_idx]),
                'predicting_items': str(history[split_idx:])
            })

# --- Convert to DataFrames ---
train_df = pd.DataFrame(train_records)
val_df = pd.DataFrame(val_records)
test_df = pd.DataFrame(test_records)

# --- Show Summary ---
print("\n--- Training Data Sample (Lower Triangular Logic) ---")
print(f"Total training examples created: {len(train_df)}")
print(train_df.head())

print("\n--- Validation Data Sample (80/20 Split) ---")
print(f"Total validation examples created: {len(val_df)}")
print(val_df.head())

print("\n--- Test Data Sample (80/20 Split) ---")
print(f"Total test examples created: {len(test_df)}")
print(test_df.head())


Total unique users: 50000
Total unique news items: 39865
Train users: 40000
Validation users: 5000
Test users: 5000

--- Training Data Sample (Lower Triangular Logic) ---
Total training examples created: 39909
   user_id                                       user_history
0        0  [6865, 16272, 24442, 31060, 27254, 5987, 29528...
1        1   [13344, 5311, 21252, 13418, 38291, 33012, 29790]
2        2  [25779, 28337, 31528, 34852, 12001, 5401, 1174...
3        3  [19793, 11314, 14564, 1881, 34663, 21541, 2775...
4        5    [31406, 18399, 29760, 4228, 14488, 3063, 12381]

--- Validation Data Sample (80/20 Split) ---
Total validation examples created: 4990
   user_id                                 given_user_history   
0        4  [9485, 13707, 24959, 15674, 39600, 18023, 2346...  \
1        6  [28618, 6754, 5759, 39737, 12825, 2228, 27671,...   
2       38  [24296, 4631, 17275, 5841, 19507, 38644, 24336...   
3       61  [28018, 21237, 27933, 4172, 11509, 7925, 10603...   
4      

In [66]:
import ast

def print_dataset_stats(name, df, history_col, predict_col=None):
    # Parse lists from string-formatted columns
    all_items = []
    for row in df.itertuples(index=False):
        history = ast.literal_eval(getattr(row, history_col))
        all_items.extend(history)
        if predict_col:
            predicted = ast.literal_eval(getattr(row, predict_col))
            all_items.extend(predicted)

    print(f"\n--- {name.upper()} SET STATS ---")
    print(f"Rows: {len(df)}")
    print(f"Unique users: {df['user_id'].nunique()}")
    print(f"Unique news items: {len(set(all_items))}")

# 📊 Print stats for all datasets
#print_dataset_stats("train", train_df, history_col='user_history')
print_dataset_stats("validation", val_df, history_col='given_user_history', predict_col='predicting_items')
print_dataset_stats("test", test_df, history_col='given_user_history', predict_col='predicting_items')



--- VALIDATION SET STATS ---
Rows: 4990
Unique users: 4990
Unique news items: 16692

--- TEST SET STATS ---
Rows: 4990
Unique users: 4990
Unique news items: 16497


In [67]:
# --- (Optional) Save mappings for reference ---
pd.Series(user2int).to_csv("dataset/user2idx.tsv", sep="\t", header=False)
pd.Series(news2int).to_csv("dataset/news2idx.tsv", sep="\t", header=False)

# --- Save final datasets ---
train_df.to_csv('dataset/training.tsv', sep='\t', index=False)
val_df.to_csv('dataset/validation.tsv', sep='\t', index=False)
test_df.to_csv('dataset/test.tsv', sep='\t', index=False)

print("Saved: training.tsv, validation.tsv, test.tsv, and mappings")

Saved: training.tsv, validation.tsv, test.tsv, and mappings


In [68]:
# Load the tsv files
train_df = pd.read_csv('dataset_impressions_concatenated/training.tsv', sep='\t')
val_df = pd.read_csv('dataset_impressions_concatenated/validation.tsv', sep='\t')
test_df = pd.read_csv('dataset_impressions_concatenated/test.tsv', sep='\t')
# Display the first few rows of each dataframe
print("Training Data:")
print(train_df.head())
print("\nValidation Data:")
print(val_df.head())    
print("\nTest Data:")
print(test_df.head())
# Display the last few rows of each dataframe
print("\nLast few rows of Training Data:")
print(train_df.tail())
print("\nLast few rows of Validation Data:")
print(val_df.tail())
print("\nLast few rows of Test Data:")
print(test_df.tail())

Training Data:
   user_id                        user_history  next_item
0     2246                             [25616]      18584
1     2246                      [25616, 18584]      14003
2     2246               [25616, 18584, 14003]      20322
3     2246        [25616, 18584, 14003, 20322]       4776
4     2246  [25616, 18584, 14003, 20322, 4776]      30188

Validation Data:
   user_id                                 given_user_history   
0    42080                                  [47, 26360, 2775]  \
1    43475  [32314, 3263, 2408, 27667, 4908, 8081, 6956, 5...   
2    17582  [11086, 508, 11009, 31456, 6832, 27845, 8175, ...   
3    25408  [25616, 3512, 23637, 2848, 489, 28284, 22745, ...   
4     3188  [12376, 28815, 27275, 27581, 565, 1230, 11472,...   

                                  predicting_items  
0                                          [13469]  
1  [21270, 9172, 6221, 21024, 30939, 16655, 17327]  
2                              [4869, 6865, 18508]  
3               

In [69]:
uid = np.array([4, 6, 38])  # Example user IDs to filter
step_next_items = val_df.loc[val_df['user_id'].isin(uid), 'predicting_items'].values
print(step_next_items)

[]


In [70]:
import ast

def validate_val_df(val_df):
    print("\n[Validation Precheck]")

    invalid_rows = 0
    total_rows = len(val_df)

    for i, row in val_df.iterrows():
        try:
            # Parse the list string into actual list (if not already a list)
            predicting_items = row['predicting_items']
            if isinstance(predicting_items, str):
                predicting_items = ast.literal_eval(predicting_items)

            if not isinstance(predicting_items, list):
                raise ValueError(f"Invalid type at row {i}")

            if len(predicting_items) == 0:
                print(f"[Warning] Empty ground truth at row {i} (user_id={row['user_id']})")
                invalid_rows += 1

        except Exception as e:
            print(f"[Error] Failed to parse row {i}: {e}")
            invalid_rows += 1

    print(f"✅ Checked {total_rows} validation rows.")
    print(f"❌ Found {invalid_rows} rows with invalid or empty ground truths.\n")

    assert (total_rows - invalid_rows) == len(val_df), \
        "Mismatch: Some validation rows are malformed. Please fix before training."

validate_val_df(val_df)



[Validation Precheck]
✅ Checked 4819 validation rows.
❌ Found 0 rows with invalid or empty ground truths.

